In [1]:
!pip install -q git+https://github.com/pbcquoc/vietocr.git --no-deps
!pip install -q einops prefetch_generator editdistance gdown pyyaml pillow torchvision

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [2]:
!pip install -q lmdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 14.4 MB/s eta 0:00:00


# VietHandOCR: VietOCR Model Fine-tuning

## 1. Cơ sở Lý thuyết & Phương pháp tiếp cận
Trong bài toán nhận dạng chữ viết tay tiếng Việt, việc huấn luyện từ đầu (from scratch) đòi hỏi tập dữ liệu khổng lồ. Chúng ta áp dụng kỹ thuật **Fine-tuning (Tinh chỉnh)**:
- **Pre-trained Weights**: Nạp bộ trọng số `vgg_transformer` đã được VietOCR tối ưu trên 10 triệu ảnh chữ in.
- **Chiến lược tối ưu**: Giữ nguyên kiến trúc Transformer Decoder, điều chỉnh các đặc trưng ở VGG Backbone để thích ứng với nét bút viết tay của tập UIT-HWDB.
- **Tốc độ học (Learning Rate)**: Đặt $\alpha = 10^{-4}$ (nhỏ hơn mức train thông thường) để tránh hiện tượng *Catastrophic Forgetting* (quên biểu diễn ngôn ngữ đã học).

In [5]:
import os

print("--- KIỂM TRA THƯ MỤC CẤP CAO /kaggle/input ---")
for ds in os.listdir('/kaggle/input'):
    print(f"📁 /kaggle/input/{ds}")
    sub_path = os.path.join('/kaggle/input', ds)
    if os.path.isdir(sub_path):
        for sub in os.listdir(sub_path):
            print(f"   └── 📁 {sub}")

--- KIỂM TRA THƯ MỤC CẤP CAO /kaggle/input ---
📁 /kaggle/input/datasets
   └── 📁 danhngnnguyn
   └── 📁 trnchihong
📁 /kaggle/input/notebooks
   └── 📁 danhngnnguyn


In [6]:
import os

DATA_ROOT = "/kaggle/input/datasets/trnchihong/viethandocr-data"
LABEL_DIR = "/kaggle/input/datasets/danhngnnguyn/results"

def inspect_annotation_fast(file_path):
    word_count, para_count, line_count = 0, 0, 0
    first_line_sample = None
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            
            rel_path = parts[0]
            if "UIT_HWDB_word" in rel_path:
                word_count += 1
            elif "UIT_HWDB_paragraph" in rel_path:
                para_count += 1
            elif "UIT_HWDB_line" in rel_path:
                line_count += 1
                if first_line_sample is None:
                    first_line_sample = rel_path
                    
    print(f"--- Thống kê: {os.path.basename(file_path)} ---")
    print(f"Số mẫu word      : {word_count}")
    print(f"Số mẫu paragraph : {para_count}")
    print(f"Số mẫu line      : {line_count}")
    
    # Chỉ kiểm tra đúng 1 file mẫu duy nhất
    if first_line_sample:
        check_path = os.path.join(DATA_ROOT, first_line_sample)
        print(f"Mẫu đường dẫn line: {first_line_sample}")
        print(f"Tồn tại trên đĩa : {os.path.isfile(check_path)}\n")

inspect_annotation_fast(os.path.join(LABEL_DIR, "train.txt"))
inspect_annotation_fast(os.path.join(LABEL_DIR, "val.txt"))

--- Thống kê: train.txt ---
Số mẫu word      : 96428
Số mẫu paragraph : 999
Số mẫu line      : 6297
Mẫu đường dẫn line: UIT_HWDB_line/UIT_HWDB_line/train_data/248/1.jpg
Tồn tại trên đĩa : True

--- Thống kê: val.txt ---
Số mẫu word      : 11179
Số mẫu paragraph : 114
Số mẫu line      : 731
Mẫu đường dẫn line: UIT_HWDB_line/UIT_HWDB_line/train_data/7/1.jpg
Tồn tại trên đĩa : True



## 2. Tiền xử lý & Lọc dữ liệu mức dòng (Line-level Dataset)
VietOCR được thiết kế tối ưu cho bài toán nhận dạng văn bản theo từng dòng đơn lẻ (Line-level OCR).
Trong tập dữ liệu hỗn hợp (gồm word, line, paragraph), chúng ta tiến hành:
1. Trích xuất riêng tập dữ liệu dòng (`UIT_HWDB_line`).
2. Chuẩn hóa chuỗi văn bản mục tiêu: loại bỏ các ký tự bọc ngoặc kép lỗi phát sinh do quá trình xuất file CSV/JSON ban đầu.
3. Chia tách thành hai file annotation sạch phục vụ quá trình huấn luyện: `train_line_clean.txt` (6.297 mẫu) và `val_line_clean.txt` (731 mẫu).

In [5]:
import os

LABEL_DIR = "/kaggle/input/datasets/danhngnnguyn/results"

def sanitize_line_annotations_fast(input_file, output_file):
    count = 0
    with open(input_file, 'r', encoding='utf-8') as fin, open(output_file, 'w', encoding='utf-8') as fout:
        for line in fin:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) < 2:
                continue
            
            rel_path, raw_label = parts[0], parts[1]
            
            # Lọc trực tiếp các dòng thuộc UIT_HWDB_line
            if 'UIT_HWDB_line' in rel_path:
                clean_label = raw_label.strip('"').replace('""', '"').strip()
                if clean_label:  # Bỏ qua nhãn rỗng
                    fout.write(f"{rel_path}\t{clean_label}\n")
                    count += 1
                    
    print(f"✓ Đã xuất {count} mẫu dòng sạch vào: {output_file}")

sanitize_line_annotations_fast(os.path.join(LABEL_DIR, 'train.txt'), '/kaggle/working/train_line_clean.txt')
sanitize_line_annotations_fast(os.path.join(LABEL_DIR, 'val.txt'), '/kaggle/working/val_line_clean.txt')

✓ Đã xuất 6297 mẫu dòng sạch vào: /kaggle/working/train_line_clean.txt
✓ Đã xuất 731 mẫu dòng sạch vào: /kaggle/working/val_line_clean.txt


In [6]:
import os
from vietocr.tool.config import Cfg

TRAIN_CLEAN = '/kaggle/working/train_line_clean.txt'
VAL_CLEAN = '/kaggle/working/val_line_clean.txt'

# 1. Nạp từ điển mặc định của VietOCR để đối chiếu
config = Cfg.load_config_from_name('vgg_transformer')
default_vocab = set(config['vocab'])


def audit_clean_text(file_path):
  print('=' * 60)
  print(f'📊 KIỂM TRA CHẤT LƯỢNG NHÃN: {os.path.basename(file_path)}')
  print('=' * 60)

  with open(file_path, 'r', encoding='utf-8') as f:
    lines = [l.strip().split('\t') for l in f if l.strip()]

  print(f'1. Tổng số dòng nhãn: {len(lines)}')

  # Kiểm tra trực quan 2 mẫu đầu và 2 mẫu cuối
  print('\n2. Mẫu 2 dòng đầu:')
  for p in lines[:2]:
    print(f'   [Path] : {p[0]}')
    print(f'   [Text] : {p[1]}')

  print('\n   Mẫu 2 dòng cuối:')
  for p in lines[-2:]:
    print(f'   [Path] : {p[0]}')
    print(f'   [Text] : {p[1]}')

  # Thống kê độ dài nhãn
  lengths = [len(p[1]) for p in lines]
  print('\n3. Thống kê độ dài chuỗi ký tự:')
  print(f'   - Ngắn nhất : {min(lengths)} ký tự')
  print(f'   - Dài nhất  : {max(lengths)} ký tự')
  print(f'   - Trung bình: {sum(lengths)/len(lengths):.1f} ký tự')

  # Kiểm định bảng ký tự so với Vocab gốc của VietOCR
  all_chars = set(''.join([p[1] for p in lines]))
  out_of_vocab = all_chars - default_vocab
  print('\n4. Kiểm định bảng chữ cái (Vocab):')
  print(f'   - Tổng ký tự độc bản : {len(all_chars)}')
  if out_of_vocab:
    print(
        f'   ⚠ Có {len(out_of_vocab)} ký tự ngoài từ điển VietOCR:'
        f' {sorted(list(out_of_vocab))}'
    )
  else:
    print('   ✓ Đạt chuẩn 100%! Toàn bộ ký tự đều nằm trong từ điển VietOCR.')
  print('\n')


audit_clean_text(TRAIN_CLEAN)
audit_clean_text(VAL_CLEAN)

📊 KIỂM TRA CHẤT LƯỢNG NHÃN: train_line_clean.txt
1. Tổng số dòng nhãn: 6297

2. Mẫu 2 dòng đầu:
   [Path] : UIT_HWDB_line/UIT_HWDB_line/train_data/248/1.jpg
   [Text] : Chuyện những người mang số 115. Sáng nay ngồi quán cà phê cóc, ông bạn già của tôi tâm sự :
   [Path] : UIT_HWDB_line/UIT_HWDB_line/train_data/248/2.jpg
   [Text] : Chú mày biết không, hôm kia không nhờ cấp cứu đã xong cái mạng già này! ".

   Mẫu 2 dòng cuối:
   [Path] : UIT_HWDB_line/UIT_HWDB_line/train_data/136/24.jpg
   [Text] : không kịp nữa, cụ đã ói tràn ra sàn xe.
   [Path] : UIT_HWDB_line/UIT_HWDB_line/train_data/136/25.jpg
   [Text] : Mùi ói xông nồng nặc.

3. Thống kê độ dài chuỗi ký tự:
   - Ngắn nhất : 3 ký tự
   - Dài nhất  : 158 ký tự
   - Trung bình: 67.7 ký tự

4. Kiểm định bảng chữ cái (Vocab):
   - Tổng ký tự độc bản : 160
   ✓ Đạt chuẩn 100%! Toàn bộ ký tự đều nằm trong từ điển VietOCR.


📊 KIỂM TRA CHẤT LƯỢNG NHÃN: val_line_clean.txt
1. Tổng số dòng nhãn: 731

2. Mẫu 2 dòng đầu:
   [Path] : UIT_HWDB

## 3. Cấu hình Huấn luyện (Training Configuration)
- **Kiến trúc Backbone & Decoder**: `vgg_transformer` (VGG19 + Transformer Decoder).
- **Học chuyển giao (Transfer Learning)**: Nạp trọng số Pretrained chính thức từ tác giả VietOCR.
- **Tốc độ học ($\alpha$)**: Đặt ở mức $10^{-4}$ với tối ưu hóa Adam, giúp điều chỉnh nhẹ trọng số VGG theo nét chữ viết tay mà không làm mất tri thức ngôn ngữ tiếng Việt của Transformer.
- **Kích thước Batch**: `batch_size = 32`, phù hợp với giới hạn bộ nhớ VRAM của GPU Tesla T4.
- **Số bước lặp (Iterations)**: Thiết lập thử nghiệm 2.000 bước ban đầu (smoke test) để đo lường độ giảm của hàm mất mát Cross-Entropy.

In [ ]:
import glob
import os
import shutil
import torch
from vietocr.model.trainer import Trainer
from vietocr.tool.config import Cfg

# 1. Bảo hiểm: Dọn dẹp cache LMDB cũ nếu còn tồn tại trong working directory
for item in glob.glob('/kaggle/working/*_hw') + glob.glob('./*_hw'):
  if os.path.exists(item):
    shutil.rmtree(item, ignore_errors=True)
    print(f'✓ Đã dọn dẹp cache cũ: {item}')

# 2. Nạp cấu hình mẫu vgg_transformer
config = Cfg.load_config_from_name('vgg_transformer')

# 3. Thiết lập phần cứng GPU
config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Thiết bị đang sử dụng: {config['device']}")

# 4. Cấu hình dataset
DATA_ROOT = '/kaggle/input/datasets/trnchihong/viethandocr-data'
config['dataset']['data_root'] = DATA_ROOT
config['dataset']['train_annotation'] = '/kaggle/working/train_line_clean.txt'
config['dataset']['valid_annotation'] = '/kaggle/working/val_line_clean.txt'

# Kích thước ảnh chuẩn hoá (thuộc nhóm dataset)
config['dataset']['image_max_width'] = 1024  # Đủ rộng cho dòng dài ~158 ký tự
config['dataset']['image_min_width'] = 32
config['dataset']['image_height'] = 32

# 5. Cấu hình DataLoader (an toàn trên Kaggle, tránh tràn bộ nhớ chia sẻ)
config['dataloader'] = {'num_workers': 0}

# 6. Đường dẫn lưu trọng số mô hình
WEIGHT_DIR = '/kaggle/working/weights'
os.makedirs(WEIGHT_DIR, exist_ok=True)
config['trainer']['save_dir'] = WEIGHT_DIR
config['trainer']['checkpoint'] = os.path.join(
    WEIGHT_DIR, 'best_val_checkpoint.pth'
)
config['trainer']['export'] = os.path.join(
    WEIGHT_DIR, 'vgg_transformer_final.pth'
)

# 7. Siêu tham số huấn luyện
config['trainer']['iters'] = 1500  # Số bước huấn luyện
config['trainer']['batch_size'] = 32  # Tối ưu cho VRAM GPU T4
config['trainer']['lr'] = 1e-4  # Tốc độ học chuẩn cho fine-tuning
config['trainer']['log_interval'] = 50  # In log sau mỗi 50 steps
config['trainer']['valid_interval'] = 300  # Đánh giá validation mỗi 300 steps
config['trainer']['print'] = True

# 8. Khởi tạo Trainer và kích hoạt quá trình Train
print('Đang khởi tạo Trainer (nạp Pretrained VGG-Transformer)...')
trainer = Trainer(config=config, pretrained=True)

print('Bắt đầu quá trình Huấn luyện!')
trainer.train()

In [ ]:
import os

def check_folder_shallow(folder_path):
    print(f"\n📂 KIỂM TRA THƯ MỤC: {folder_path}")
    if not os.path.exists(folder_path):
        print("   ⚠ Thư mục này không tồn tại!")
        return
    
    # Chỉ đọc 1 cấp duy nhất, không đệ quy sâu
    with os.scandir(folder_path) as entries:
        items = list(entries)
        if not items:
            print("   (Thư mục trống)")
            return
        
        for entry in items:
            if entry.is_file():
                size_mb = entry.stat().st_size / (1024 * 1024)
                print(f"   📄 [File] {entry.name:<30} ({size_mb:.2f} MB)")
            elif entry.is_dir():
                print(f"   📁 [Folder] {entry.name}/")

# 1. Kiểm tra cấp ngoài cùng của /kaggle/working
check_folder_shallow('/kaggle/working')

# 2. Kiểm tra trực tiếp thư mục con weights (nếu có)
check_folder_shallow('/kaggle/working/weights')

In [ ]:
import os

LOG_FILE = '/kaggle/working/train.log'

if os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"=== TỔNG SỐ DÒNG TRONG TRAIN.LOG: {len(lines)} ===")
    print("--- 20 DÒNG CUỐI CÙNG TRONG LOG ---")
    for line in lines[-20:]:
        print(line.strip())
else:
    print("Không tìm thấy file train.log!")

## 4. Lưu trữ Mô hình & Trọng số Huấn luyện (Model Checkpointing)

Quá trình fine-tuning mô hình `vgg_transformer` đã hoàn tất qua 1.500 bước lặp với giá trị hàm mất mát (Cross-Entropy Loss) giảm đều từ **1.190 xuống 0.832**.

Do cấu chế xuất file tự động của VietOCR phụ thuộc vào đường dẫn thư mục `export`, để đảm bảo tính toàn vẹn dữ liệu và tránh thất thoát mô hình sau khi kết thúc phiên tính toán:
- Trích xuất trực tiếp `state_dict` từ đối tượng mô hình nơ-ron `trainer.model` đang lưu trữ trong RAM GPU/CPU.
- Lưu cấu trúc trọng số đã học vào file nhị phân PyTorch `.pth` tại thư mục `/kaggle/working/weights/vgg_transformer_manual_saved.pth`.

In [ ]:
import os
import torch

# 1. Định nghĩa thư mục lưu trữ weights
WEIGHT_DIR = "/kaggle/working/weights"
os.makedirs(WEIGHT_DIR, exist_ok=True)
TARGET_WEIGHT_PATH = os.path.join(WEIGHT_DIR, "vgg_transformer_manual_saved.pth")

# 2. Trích xuất và lưu state_dict trực tiếp từ đối tượng trainer trong bộ nhớ
if 'trainer' in globals() and hasattr(trainer, 'model'):
    torch.save(trainer.model.state_dict(), TARGET_WEIGHT_PATH)
    file_size_mb = os.path.getsize(TARGET_WEIGHT_PATH) / (1024 * 1024)
    print(f"✓ Đã lưu thành công trọng số mô hình: {TARGET_WEIGHT_PATH}")
    print(f"✓ Dung lượng file: {file_size_mb:.2f} MB")
else:
    raise RuntimeError("Không tìm thấy biến 'trainer' hoặc 'trainer.model' trong bộ nhớ RAM!")

## 5. Trực quan hóa & Đánh giá Định tính (Qualitative Evaluation)

Để kiểm chứng năng lực giải mã thực tế của mạng Transformer sau khi học chuyển giao nét chữ viết tay từ tập dữ liệu `UIT-HWDB_line`:
1. **Khởi tạo Bộ suy luận (Inference Pipeline)**: Nạp kiến trúc `vgg_transformer` và liên kết với file trọng số vừa xuất `vgg_transformer_manual_saved.pth`.
2. **Kiểm thử mẫu ngẫu nhiên**: Lấy ngẫu nhiên các dòng văn bản từ tập kiểm định độc lập `val_line_clean.txt`.
3. **So sánh Định tính (Qualitative Comparison)**:
   - Trực quan hóa hình ảnh dòng chữ viết tay gốc.
   - Đối chiếu nhãn thực tế (*Ground Truth*) và kết quả mô hình nhận dạng (*Prediction*).
   - Xuất đồ thị kiểm chứng thành file ảnh độ phân giải cao `prediction_visualization.png`.

In [ ]:
import os
import random
from PIL import Image
import matplotlib.pyplot as plt
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

# 1. Cấu hình pipeline suy luận (Inference)
eval_config = Cfg.load_config_from_name('vgg_transformer')
eval_config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
eval_config['weights'] = '/kaggle/working/weights/vgg_transformer_manual_saved.pth'
eval_config['predictor']['beamsearch'] = False

print(f"Đang khởi tạo Predictor trên thiết bị: {eval_config['device']}")
detector = Predictor(eval_config)

# 2. Đọc danh sách nhãn kiểm định
VAL_CLEAN = '/kaggle/working/val_line_clean.txt'
DATA_ROOT = "/kaggle/input/datasets/trnchihong/viethandocr-data"

with open(VAL_CLEAN, 'r', encoding='utf-8') as f:
    val_items = [line.strip().split('\t') for line in f if line.strip()]

# 3. Lấy ngẫu nhiên 4 mẫu để kiểm tra trực quan
random.seed(42)
sample_items = random.sample(val_items, 4)

print("\n" + "=" * 70)
print("🔍 KẾT QUẢ NHẬN DẠNG THỰC TẾ TRÊN TẬP VALIDATION")
print("=" * 70)

# Khởi tạo khung vẽ đồ thị Matplotlib
fig, axes = plt.subplots(4, 1, figsize=(14, 10))

for idx, (rel_path, ground_truth) in enumerate(sample_items):
    full_path = os.path.join(DATA_ROOT, rel_path)
    img = Image.open(full_path)
    
    # Dự đoán chuỗi ký tự qua VietOCR Predictor
    pred_text = detector.predict(img)
    
    # Đánh giá mức độ trùng khớp
    is_match = ground_truth.strip() == pred_text.strip()
    status_label = "✓ CHÍNH XÁC HOÀN TOÀN" if is_match else "⚠ CÓ KHÁC BIỆT"
    
    print(f"\n[Mẫu {idx + 1}] Đường dẫn: {rel_path}")
    print(f"  - Nhãn gốc (GT)  : {ground_truth}")
    print(f"  - Mô hình đoán   : {pred_text}")
    print(f"  - Đánh giá       : {status_label}")

    # Vẽ dòng chữ viết tay lên subplot
    axes[idx].imshow(img)
    axes[idx].axis('off')
    title_color = '#1b7837' if is_match else '#b2182b'
    axes[idx].set_title(
        f"GT:   {ground_truth}\nPRED: {pred_text}", 
        fontsize=10, 
        loc='left', 
        color=title_color, 
        pad=6
    )

plt.tight_layout()
output_chart = '/kaggle/working/prediction_visualization.png'
plt.savefig(output_chart, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Đã lưu file biểu đồ trực quan hóa: {output_chart}")

## 6. Cơ chế Đo lường Chỉ số: CER và WER qua Ví dụ Thực tế

Để đo độ sai lệch giữa chuỗi dự đoán (**Hypothesis**) và chuỗi gốc (**Reference**), ta sử dụng thuật toán khoảng cách Levenshtein:
- **CER (Character Error Rate)**:
  - So sánh trực tiếp danh sách ký tự: `list("con mèo")` vs `list("con mẻo")`.
  - Khoảng cách Levenshtein = 1 (sai 1 ký tự `è` $\rightarrow$ `ẻ`).
  - $CER = \frac{1}{7} \approx 14.3\%$.
- **WER (Word Error Rate)**:
  - Tách chuỗi thành danh sách từ: `["con", "mèo"]` vs `["con", "mẻo"]`.
  - Khoảng cách Levenshtein = 1 (sai cụm từ `"mèo"` $\rightarrow$ `"mẻo"`).
  - $WER = \frac{1}{2} = 50.0\%$.

Đoạn code dưới đây định nghĩa hàm tính toán chuẩn xác bằng thư viện C++ `editdistance` đã cài đặt sẵn.

In [ ]:
import editdistance

def calculate_cer(reference: str, hypothesis: str) -> float:
    """
    Tính Tỷ lệ Lỗi Ký tự (CER) giữa 2 chuỗi.
    """
    ref_chars = list(reference)
    hyp_chars = list(hypothesis)
    if len(ref_chars) == 0:
        return 0.0 if len(hyp_chars) == 0 else 1.0
    dist = editdistance.eval(ref_chars, hyp_chars)
    return dist / len(ref_chars)

def calculate_wer(reference: str, hypothesis: str) -> float:
    """
    Tính Tỷ lệ Lỗi Từ (WER) giữa 2 chuỗi.
    """
    ref_words = reference.strip().split()
    hyp_words = hypothesis.strip().split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    dist = editdistance.eval(ref_words, hyp_words)
    return dist / len(ref_words)

# Chạy thử nghiệm trên chính Mẫu 1 trong log vừa rồi của bạn:
ref_sample = "đập đi đập lại chỉ còn chú lơ thuyền và tôi."
hyp_sample = "đập đi đập lại chỉ còn chú lở thuyền và tôi."

sample_cer = calculate_cer(ref_sample, hyp_sample)
sample_wer = calculate_wer(ref_sample, hyp_sample)

print("--- MINH HỌA CƠ CHẾ TÍNH LỖI ---")
print(f"Chuỗi gốc (Ref) : '{ref_sample}'")
print(f"Chuỗi đoán (Hyp): '{hyp_sample}'")
print(f"-> Số ký tự gốc  : {len(ref_sample)}")
print(f"-> Số từ gốc     : {len(ref_sample.split())}")
print(f"-> CER           : {sample_cer:.4f} ({sample_cer*100:.2f}%)  [Chỉ sai 1 dấu]")
print(f"-> WER           : {sample_wer:.4f} ({sample_wer*100:.2f}%)  [Sai 1 trên 11 từ]")

## 7. Đánh giá Định lượng Toàn diện trên Tập Validation

Tiến hành suy luận trên toàn bộ 731 mẫu dòng của tập kiểm định `val_line_clean.txt`:
1. **Duyệt qua từng mẫu ảnh**: Đoán chuỗi ký tự qua mô hình fine-tuned.
2. **Tổng hợp Chỉ số Hệ thống**:
   - $\text{CER}_{\text{corpus}} = \frac{\sum \text{Khoảng cách Levenshtein Ký tự}}{\sum \text{Tổng số ký tự Ground Truth}}$
   - $\text{WER}_{\text{corpus}} = \frac{\sum \text{Khoảng cách Levenshtein Từ}}{\sum \text{Tổng số từ Ground Truth}}$
   - **Line Accuracy (Exact Match)**: Tỷ lệ dòng khớp chính xác 100%.
3. **Trích xuất Mẫu lỗi điển hình**: Phân tích các trường hợp có khoảng cách sai lệch lớn nhất để làm tiền đề cải tiến mô hình.

In [ ]:
import os
import editdistance
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd

VAL_CLEAN = '/kaggle/working/val_line_clean.txt'
DATA_ROOT = "/kaggle/input/datasets/trnchihong/viethandocr-data"

# Đọc toàn bộ nhãn kiểm định
with open(VAL_CLEAN, 'r', encoding='utf-8') as f:
    eval_samples = [line.strip().split('\t') for line in f if line.strip()]

total_char_dist = 0
total_char_ref_len = 0

total_word_dist = 0
total_word_ref_len = 0

exact_match_count = 0
evaluation_records = []

print(f"Bắt đầu đánh giá trên toàn bộ {len(eval_samples)} mẫu Validation...")

# Duyệt tuần tự an toàn với tqdm đo tiến độ (chạy trên GPU khoảng 1-2 phút)
for rel_path, ground_truth in tqdm(eval_samples, desc="Evaluating"):
    full_path = os.path.join(DATA_ROOT, rel_path)
    if not os.path.exists(full_path):
        continue
        
    img = Image.open(full_path)
    pred_text = detector.predict(img).strip()
    gt_text = ground_truth.strip()
    
    # 1. Đo mức độ cấp Ký tự
    c_dist = editdistance.eval(list(gt_text), list(pred_text))
    c_len = len(gt_text)
    total_char_dist += c_dist
    total_char_ref_len += c_len
    cer_sample = c_dist / c_len if c_len > 0 else 0.0
    
    # 2. Đo mức độ cấp Từ
    w_gt = gt_text.split()
    w_pred = pred_text.split()
    w_dist = editdistance.eval(w_gt, w_pred)
    w_len = len(w_gt)
    total_word_dist += w_dist
    total_word_ref_len += w_len
    wer_sample = w_dist / w_len if w_len > 0 else 0.0
    
    # 3. Khớp chính xác 100%
    is_exact = (gt_text == pred_text)
    if is_exact:
        exact_match_count += 1
        
    evaluation_records.append({
        'path': rel_path,
        'ground_truth': gt_text,
        'prediction': pred_text,
        'char_dist': c_dist,
        'cer': cer_sample,
        'word_dist': w_dist,
        'wer': wer_sample,
        'exact_match': is_exact
    })

# Tính toán các chỉ số tổng thể toàn tập (Corpus-level)
corpus_cer = total_char_dist / total_char_ref_len if total_char_ref_len > 0 else 0.0
corpus_wer = total_word_dist / total_word_ref_len if total_word_ref_len > 0 else 0.0
accuracy = (exact_match_count / len(evaluation_records)) * 100

print("\n" + "="*60)
print("📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ (VALIDATION BENCHMARK)")
print("="*60)
print(f"Tổng số mẫu kiểm định       : {len(evaluation_records)}")
print(f"Số dòng đúng hoàn toàn (100%): {exact_match_count}/{len(evaluation_records)}")
print(f"Độ chính xác dòng (Accuracy): {accuracy:.2f}%")
print(f"Tỷ lệ lỗi ký tự (CER)       : {corpus_cer*100:.2f}% (Chỉ số: {corpus_cer:.4f})")
print(f"Tỷ lệ lỗi từ (WER)          : {corpus_wer*100:.2f}% (Chỉ số: {corpus_wer:.4f})")
print("="*60)

# Chuyển kết quả sang DataFrame để dễ quan sát và lưu file
df_eval = pd.DataFrame(evaluation_records)
df_eval.to_csv('/kaggle/working/validation_evaluation_results.csv', index=False, encoding='utf-8-sig')
print("✓ Đã lưu chi tiết đánh giá từng mẫu vào: /kaggle/working/validation_evaluation_results.csv")

## 8. Phân tích Trực quan Chuyên sâu: Top 5 Mẫu Lỗi Lớn Nhất & Top 5 Mẫu Hoàn Hảo Nhất (Kèm Đường dẫn Truy vết)

Phân tích trực quan 10 mẫu tiêu biểu từ tập kiểm định kèm đường dẫn truy vết trên đĩa:
1. **Top 5 Mẫu CER Cao Nhất**: Rà soát các trường hợp sai lệch lớn để truy vết file gốc trong dataset, phân biệt lỗi do nhãn gốc (Truncated Ground Truth) hay do nét chữ.
2. **Top 5 Mẫu Hoàn Hảo Dài Nhất**: Kiểm chứng năng lực nhận dạng chuỗi dài trên các ảnh cụ thể trong tập dữ liệu.

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

EVAL_CSV = '/kaggle/working/validation_evaluation_results.csv'
DATA_ROOT = '/kaggle/input/datasets/trnchihong/viethandocr-data'

if not os.path.exists(EVAL_CSV):
  raise FileNotFoundError(f'Không tìm thấy file: {EVAL_CSV}')

df_results = pd.read_csv(EVAL_CSV)

# 1. Trích xuất Top 5 mẫu lỗi nặng nhất và Top 5 mẫu đúng dài nhất
worst_5_samples = df_results.sort_values(by='cer', ascending=False).head(5)
best_5_samples = (
    df_results[df_results['exact_match'] == True]
    .sort_values(by='ground_truth', key=lambda x: x.str.len(), ascending=False)
    .head(5)
)


def plot_top_samples_with_paths(samples_df, title_text, is_error=True):
  print('=' * 90)
  print(f'📊 {title_text.upper()}')
  print('=' * 90)

  n_samples = len(samples_df)
  fig, axes = plt.subplots(n_samples, 1, figsize=(16, 3.2 * n_samples))
  if n_samples == 1:
    axes = [axes]

  for idx, (_, row) in enumerate(samples_df.iterrows()):
    rel_path = row['path']
    full_path = os.path.join(DATA_ROOT, rel_path)
    gt = str(row['ground_truth'])
    pred = str(row['prediction'])
    cer_val = row['cer'] * 100

    # In thông tin chi tiết kèm đường dẫn tuyệt đối để copy truy vết
    prefix = 'MẪU LỖI' if is_error else 'MẪU CHUẨN'
    print(f'\n[{prefix} {idx + 1}]')
    print(f'  📁 Relative Path : {rel_path}')
    print(f'  🔗 Full Path     : {full_path}')
    print(f'  - GT (Thực tế)   : {gt}')
    print(f'  - PRED (Dự đoán) : {pred}')
    if is_error:
      print(
          f"  - CER            : {cer_val:.2f}% (Lỗi"
          f" {row['char_dist']}/{len(gt)} ký tự)"
      )
    else:
      print(f'  - Độ dài dòng    : {len(gt)} ký tự (Khớp 100%)')

    # Đọc và hiển thị ảnh
    if os.path.exists(full_path):
      img = Image.open(full_path)
      axes[idx].imshow(img)
    axes[idx].axis('off')

    # Hiển thị trên biểu đồ cả thông tin nhãn lẫn đường dẫn file
    header_color = '#b2182b' if is_error else '#1b7837'
    info_label = (
        f'[{idx + 1}] File: {rel_path}\n'
        f'GT:   {gt}\n'
        f'PRED: {pred}'
        + (f'  [CER: {cer_val:.1f}%]' if is_error else '  [Exact Match 100%]')
    )
    axes[idx].set_title(
        info_label, fontsize=9.5, loc='left', color=header_color, pad=6
    )

  plt.tight_layout()
  save_path = (
      f"/kaggle/working/{'top5_worst_errors' if is_error else 'top5_best_matches'}.png"
  )
  plt.savefig(save_path, dpi=300, bbox_inches='tight')
  plt.show()
  print(f'\n✓ Đã xuất ảnh biểu đồ: {save_path}\n')


# 2. Thực thi hiển thị
plot_top_samples_with_paths(
    worst_5_samples,
    'Top 5 Mẫu Có Tỷ Lệ Lỗi Ký Tự Cao Nhất (Worst CER)',
    is_error=True,
)
plot_top_samples_with_paths(
    best_5_samples,
    'Top 5 Mẫu Dòng Dài Nhất Được Nhận Dạng Đúng 100% (Best Matches)',
    is_error=False,
)

## 8.5. Trực quan hóa Tiến trình Huấn luyện từ File Log (Training Curves)

Bóc tách dữ liệu lịch sử từ file `/kaggle/working/train.log` để trực quan hóa:
1. **Đường cong mất mát (Loss Curve)**: Đánh giá độ hội tụ và tốc độ giảm mất mát của mô hình VGG-Transformer từ bước 200 đến 1.400.
2. **Tốc độ học (Learning Rate Decay)**: Xác nhận chiến lược suy giảm tốc độ học (Cosine Annealing Schedule).
3. **Phân bổ thời gian thực thi (GPU Time vs Data Load Time)**: Kiểm tra tính ổn định của luồng dữ liệu trên môi trường máy ảo Kaggle.

In [ ]:
import os
import re
import matplotlib.pyplot as plt

LOG_FILE = '/kaggle/working/train.log'

if not os.path.exists(LOG_FILE):
  print(f'⚠ Không tìm thấy file {LOG_FILE} để trích xuất biểu đồ!')
else:
  # 1. Trích xuất thông số bằng Regular Expression
  pattern = re.compile(
      r'iter:\s*(\d+)\s*-\s*train loss:\s*([\d\.]+)\s*-\s*lr:\s*([\d\.e\-\+]+)\s*-\s*load'
      r' time:\s*([\d\.]+)\s*-\s*gpu time:\s*([\d\.]+)'
  )

  iters = []
  losses = []
  lrs = []
  load_times = []
  gpu_times = []

  with open(LOG_FILE, 'r', encoding='utf-8') as f:
    for line in f:
      match = pattern.search(line)
      if match:
        iters.append(int(match.group(1)))
        losses.append(float(match.group(2)))
        lrs.append(float(match.group(3)))
        load_times.append(float(match.group(4)))
        gpu_times.append(float(match.group(5)))

  print(f'✓ Đã trích xuất thành công {len(iters)} điểm ghi nhận tiến trình.')

  # 2. Khởi tạo đồ thị đa khung (3 Subplots)
  fig, axes = plt.subplots(1, 3, figsize=(18, 5))

  # Biểu đồ 1: Train Loss
  axes[0].plot(
      iters, losses, marker='o', color='#1f77b4', linewidth=2, markersize=6
  )
  axes[0].set_title(
      'Hàm mất mát huấn luyện (Train Loss)', fontsize=12, fontweight='bold'
  )
  axes[0].set_xlabel('Số bước lặp (Iterations)', fontsize=11)
  axes[0].set_ylabel('Cross-Entropy Loss', fontsize=11)
  axes[0].grid(True, linestyle='--', alpha=0.6)
  for x, y in zip(iters, losses):
    axes[0].annotate(
        f'{y:.3f}',
        (x, y),
        textcoords='offset points',
        xytext=(0, 7),
        ha='center',
        fontsize=9,
    )

  # Biểu đồ 2: Learning Rate Schedule
  axes[1].plot(
      iters, lrs, marker='s', color='#2ca02c', linewidth=2, markersize=6
  )
  axes[1].set_title(
      'Tốc độ học (Learning Rate Decay)', fontsize=12, fontweight='bold'
  )
  axes[1].set_xlabel('Số bước lặp (Iterations)', fontsize=11)
  axes[1].set_ylabel('Learning Rate', fontsize=11)
  axes[1].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
  axes[1].grid(True, linestyle='--', alpha=0.6)

  # Biểu đồ 3: GPU Time vs Data Load Time
  axes[2].plot(
      iters,
      gpu_times,
      marker='^',
      color='#d62728',
      linewidth=1.8,
      label='GPU Compute Time (s)',
  )
  axes[2].plot(
      iters,
      load_times,
      marker='v',
      color='#ff7f0e',
      linewidth=1.8,
      label='Data Load Time (s)',
  )
  axes[2].set_title(
      'Thời gian thực thi mỗi 200 steps (s)', fontsize=12, fontweight='bold'
  )
  axes[2].set_xlabel('Số bước lặp (Iterations)', fontsize=11)
  axes[2].set_ylabel('Giây (Seconds)', fontsize=11)
  axes[2].legend(loc='best', frameon=True)
  axes[2].grid(True, linestyle='--', alpha=0.6)

  plt.tight_layout()
  plot_output = '/kaggle/working/training_progress_metrics.png'
  plt.savefig(plot_output, dpi=300, bbox_inches='tight')
  plt.show()

  print(f'✓ Đã lưu biểu đồ theo dõi quá trình train vào: {plot_output}')

## 9. Dọn dẹp Bộ nhớ Đệm & Bảo toàn Tài sản Xuất Output (Clean & Export Preparation)

Trước khi lưu trữ phiên bản notebook (Save Version) và liên kết mã nguồn với GitHub:
1. **Giải phóng thư mục đệm LMDB tạm thời**: Loại bỏ các thư mục đệm `train_data/`, `valid_data/` và `.virtual_documents/` để giải phóng dung lượng đĩa và đẩy nhanh tốc độ nén output của Kaggle.
2. **Kiểm kê & Bảo toàn toàn bộ sản phẩm đầu ra**:
   - Trọng số mô hình đã tinh chỉnh: `weights/vgg_transformer_manual_saved.pth` (~145 MB).
   - Nhật ký huấn luyện: `train.log`.
   - Kết quả đánh giá chi tiết: `validation_evaluation_results.csv` (731 mẫu với CER, WER).
   - Toàn bộ đồ thị trực quan hóa:
     - `training_progress_metrics.png`: Diễn biến Train Loss, Learning Rate và thời gian GPU/Load.
     - `top5_worst_errors.png`: 5 mẫu có tỷ lệ lỗi ký tự cao nhất kèm nhãn và đường dẫn truy vết.
     - `top5_best_matches.png`: 5 mẫu dòng dài đạt độ chính xác tuyệt đối 100%.
     - `prediction_visualization.png`: Trực quan hóa suy luận thực tế.

In [ ]:
import os
import shutil

# 1. Danh sách các thư mục đệm tạm cần xóa bỏ để tối ưu dung lượng
temp_dirs = [
    '/kaggle/working/train_data',
    '/kaggle/working/valid_data',
    '/kaggle/working/.virtual_documents',
]

print("--- TIẾN HÀNH DỌN DẸP BỘ NHỚ ĐỆM TẠM ---")
for d in temp_dirs:
  if os.path.exists(d):
    shutil.rmtree(d, ignore_errors=True)
    print(f"✓ Đã xóa thư mục đệm: {d}")

# 2. Liệt kê toàn bộ các file thành phẩm sẽ được lưu vào Output của Kaggle
print("\n" + "=" * 70)
print("📦 DANH MỤC TÀI SẢN HOÀN CHỈNH SẴN SÀNG LƯU TRỮ OUTPUT")
print("=" * 70)

total_size_mb = 0.0

for root, dirs, files in os.walk('/kaggle/working'):
  for f in sorted(files):
    f_path = os.path.join(root, f)
    rel_path = os.path.relpath(f_path, '/kaggle/working')
    size_mb = os.path.getsize(f_path) / (1024 * 1024)
    total_size_mb += size_mb
    print(f"📄 {rel_path:<45} ({size_mb:6.2f} MB)")

print("-" * 70)
print(f"Tổng dung lượng bộ tài sản cần lưu trữ: {total_size_mb:.2f} MB")
print("=" * 70)
print("✓ Quá trình chuẩn bị hoàn tất!")
print('👉 Bây giờ bạn có thể bấm "Save Version" (chọn Quick Save + Save output).')

In [1]:
import os

# Chỉ xem tên các thư mục gốc trong datasets, không quét vào trong
all_datasets = os.listdir('/kaggle/input/datasets')
# Lọc bỏ dataset ảnh 'trnchihong' để tuyệt đối không kích hoạt tải ảnh
checkpoint_candidates = [d for d in all_datasets if d != 'trnchihong']
print("Dataset khả dụng để tìm weights:", checkpoint_candidates)

Dataset khả dụng để tìm weights: ['danhngnnguyn']


In [2]:
import os

# Chỉ xem tên các thư mục gốc trong datasets, không quét vào trong
all_datasets = os.listdir('/kaggle/input/datasets')
# Lọc bỏ dataset ảnh 'trnchihong' để tuyệt đối không kích hoạt tải ảnh
checkpoint_candidates = [d for d in all_datasets if d != 'trnchihong']
print("Dataset khả dụng để tìm weights:", checkpoint_candidates)

Dataset khả dụng để tìm weights: ['danhngnnguyn']


In [3]:
import os
print("Các mục con bên trong danhngnnguyn:")
print(os.listdir('/kaggle/input/datasets/danhngnnguyn'))

Các mục con bên trong danhngnnguyn:
['vietocr-v1-checkpoint', 'results']


In [4]:
import os
print(os.listdir('/kaggle/input/datasets/danhngnnguyn/vietocr-v1-checkpoint'))

['train.log', 'weights', 'train_line_clean.txt', 'training_progress_metrics.png', 'prediction_visualization.png', 'val_line_clean.txt', 'validation_evaluation_results.csv', 'best_matches_viz.png', 'top5_worst_errors.png', 'worst_errors_viz.png', 'top5_best_matches.png']


In [5]:
import os
print(os.listdir('/kaggle/input/datasets/danhngnnguyn/vietocr-v1-checkpoint/weights'))

['vgg_transformer_manual_saved.pth']


## 10. Khảo sát Định lượng Nhanh trên Tập Huấn luyện (Train Set Sanity Check - 200 Mẫu)

Mục tiêu chẩn đoán hiện tượng Underfitting/Overfitting (Bias - Variance):
- Đo lường Train CER, Train WER và Train Accuracy trên 200 mẫu ngẫu nhiên từ `train_line_clean.txt`.
- Đối chiếu với kết quả Validation Baseline (CER Val = 8.18%, WER Val = 21.91%) để hoàn thiện bức tranh chẩn đoán trước khi chuyển giao sang notebook mới.

In [7]:
import os
import random
import editdistance
from PIL import Image
from tqdm.auto import tqdm
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

# 1. Đường dẫn file weights đã xác thực chính xác
WEIGHT_PATH = "/kaggle/input/datasets/danhngnnguyn/vietocr-v1-checkpoint/weights/vgg_transformer_manual_saved.pth"
print(f"✓ Đang nạp weights từ: {WEIGHT_PATH}")

# 2. Khởi tạo Predictor
eval_config = Cfg.load_config_from_name('vgg_transformer')
eval_config['device'] = 'cuda:0'
eval_config['weights'] = WEIGHT_PATH
eval_config['predictor']['beamsearch'] = False

detector = Predictor(eval_config)

# 3. Lấy 200 mẫu ngẫu nhiên từ tập Train sạch
TRAIN_CLEAN = '/kaggle/working/train_line_clean.txt'
DATA_ROOT = "/kaggle/input/datasets/trnchihong/viethandocr-data"

with open(TRAIN_CLEAN, 'r', encoding='utf-8') as f:
    all_train_samples = [line.strip().split('\t') for line in f if line.strip()]

random.seed(42)
sample_train_200 = random.sample(all_train_samples, min(200, len(all_train_samples)))

print(f"Bắt đầu khảo sát trên {len(sample_train_200)} mẫu Train...")

total_char_dist = 0
total_char_len = 0
total_word_dist = 0
total_word_len = 0
exact_match_count = 0
valid_samples = 0

for rel_path, ground_truth in tqdm(sample_train_200, desc="Evaluating Train"):
    full_path = f"{DATA_ROOT}/{rel_path}"
    
    try:
        with Image.open(full_path) as img:
            pred_text = detector.predict(img).strip()
    except (FileNotFoundError, OSError):
        continue
    
    gt_text = ground_truth.strip()
    valid_samples += 1
    
    # Đo cấp Ký tự (CER)
    c_dist = editdistance.eval(list(gt_text), list(pred_text))
    total_char_dist += c_dist
    total_char_len += len(gt_text)
    
    # Đo cấp Từ (WER)
    w_gt = gt_text.split()
    w_pred = pred_text.split()
    w_dist = editdistance.eval(w_gt, w_pred)
    total_word_dist += w_dist
    total_word_len += len(w_gt)
    
    if gt_text == pred_text:
        exact_match_count += 1

train_sample_cer = total_char_dist / total_char_len if total_char_len > 0 else 0.0
train_sample_wer = total_word_dist / total_word_len if total_word_len > 0 else 0.0
train_acc = (exact_match_count / valid_samples) * 100 if valid_samples > 0 else 0.0

print("\n" + "=" * 60)
print(f"📊 KẾT QUẢ KHẢO SÁT TẬP TRAIN ({valid_samples}/200 MẪU)")
print("=" * 60)
print(f"Chính xác dòng (Accuracy) : {train_acc:.2f}% ({exact_match_count}/{valid_samples})")
print(f"Tỷ lệ lỗi ký tự (CER)    : {train_sample_cer*100:.2f}% (Chỉ số: {train_sample_cer:.4f})")
print(f"Tỷ lệ lỗi từ (WER)       : {train_sample_wer*100:.2f}% (Chỉ số: {train_sample_wer:.4f})")
print("=" * 60)
print(f"👉 Đối chiếu Validation Baseline (Hôm qua): CER = 8.18% | WER = 21.91%")

✓ Đang nạp weights từ: /kaggle/input/datasets/danhngnnguyn/vietocr-v1-checkpoint/weights/vgg_transformer_manual_saved.pth
Bắt đầu khảo sát trên 200 mẫu Train...


Evaluating Train:   0%|          | 0/200 [00:00<?, ?it/s]


📊 KẾT QUẢ KHẢO SÁT TẬP TRAIN (200/200 MẪU)
Chính xác dòng (Accuracy) : 24.50% (49/200)
Tỷ lệ lỗi ký tự (CER)    : 5.14% (Chỉ số: 0.0514)
Tỷ lệ lỗi từ (WER)       : 13.68% (Chỉ số: 0.1368)
👉 Đối chiếu Validation Baseline (Hôm qua): CER = 8.18% | WER = 21.91%
